In [1]:
from aiida import orm, load_profile
from aiida.storage.sqlite_zip.backend import SqliteZipBackend
from aiida_wannier90_workflows.workflows.optimize import Wannier90OptimizeWorkChain
from tabulate import tabulate

In [2]:
archive_profile = SqliteZipBackend.create_profile("mc3d_wannier_3materials.aiida")
load_profile(archive_profile)

Profile<uuid='703bac82764c4b44b3782ab167261deb' name='mc3d_wannier_3materials.aiida'>

In [4]:
w90_workchains = orm.QueryBuilder().append(Wannier90OptimizeWorkChain).all(flat=True)
w90_workchains

[<WorkChainNode: uuid: 94422a15-d26d-4d95-a0c8-9b4fb7f07e91 (pk: 6607169) (aiida.workflows:wannier90_workflows.optimize)>,
 <WorkChainNode: uuid: d45b0108-da1f-40d6-98ad-0f9d67186e43 (pk: 9892286) (aiida.workflows:wannier90_workflows.optimize)>,
 <WorkChainNode: uuid: 3ee04a62-2d61-4f0e-8cee-cb77967ef330 (pk: 13171598) (aiida.workflows:wannier90_workflows.optimize)>]

In [5]:
data = []
for w90 in w90_workchains:
    # This is the BandsData for Wannier90
    band_w90 = w90.outputs.band_structure

    # This is PwBandsWorkChain
    pwband = w90.inputs.nscf.pw.parent_folder.creator.inputs.stash_data.creator.caller.caller
    # This is the BandsData
    band_pw = pwband.outputs.band_structure

    # This is the structure, can be linked to MC3D, e.g.
    # https://mc3d.materialscloud.org/details/mc3d-43521/pbesol-v1
    structure = pwband.inputs.structure

    # This is the path to stashed files
    stash = w90.outputs.wannier90_plot.remote_stash.base.attributes.get('target_basepath')
    # Remove the prefix folder
    stash = stash.split("wannier/")[1]
    # Each folder contains a wan_tb_cube.7z file, which contains the real-space
    # Wannier functions in cube files, and the tight-binding model in
    # aiida_tb.dat and aiida_wsvec.dat files.

    data.append({
        "uuid_w90": w90.uuid,
        "formula_hill": w90.base.extras.get('formula_hill'),
        "number_of_sites": w90.base.extras.get('number_of_sites'),
        "source_db": w90.base.extras.get('source_db'),
        "source_id": w90.base.extras.get('source_id'),
        "uuid_pwband": pwband.uuid,
        "uuid_structure": structure.uuid,
        "uuid_band_pw": band_pw.uuid,
        "uuid_band_w90": band_w90.uuid,
        "stash_folder": stash,
    })

print(tabulate(data, headers="keys"))

uuid_w90                              formula_hill      number_of_sites  source_db    source_id    uuid_pwband                           uuid_structure                        uuid_band_pw                          uuid_band_w90                         stash_folder
------------------------------------  --------------  -----------------  -----------  -----------  ------------------------------------  ------------------------------------  ------------------------------------  ------------------------------------  --------------------------------------
94422a15-d26d-4d95-a0c8-9b4fb7f07e91  H2S3                            5  icsd         672476       fee3ed65-c9bb-4508-a09c-4a46fe841487  23c8d6c6-b10f-4268-99bd-92129523f110  94f5bca9-da7c-425f-83a4-500ba2f65d40  50352909-fad1-41da-9549-538d0a1bde81  60/08/011b-45de-467d-aecd-b2f09f22cd34
d45b0108-da1f-40d6-98ad-0f9d67186e43  As4Li4O20Ti4                   32  icsd         172581       87316327-3dd0-4da7-a8d1-bde3f0bb4eb7  3484da06-ed6e-4fca-

In [7]:
# For each Wannier90OptimizeWorkChain, it contains tens to hundreds of Wannier functions.
w90 = w90_workchains[0]
out_dict = w90.outputs.wannier90_optimal.output_parameters.get_dict()
out_dict

{'Omega_D': 0.026051159,
 'Omega_I': 16.310849001,
 'Omega_OD': 2.331212402,
 'warnings': [],
 'number_wfs': 14,
 'length_units': 'Ang',
 'xyz_writeout': 'F',
 'r2mn_writeout': 'F',
 'preprocess_only': 'F',
 'output_verbosity': 1,
 'convergence_tolerance': 5e-07,
 'wannier_functions_output': [{'wf_ids': 1,
   'wf_centres': [1.38448, 4.069295, 1.773057],
   'wf_spreads': 1.32288518},
  {'wf_ids': 2,
   'wf_centres': [-0.919722, 2.009378, 4.143696],
   'wf_spreads': 1.32296408},
  {'wf_ids': 3,
   'wf_centres': [4.414502, 0.245519, 4.320885],
   'wf_spreads': 1.31446042},
  {'wf_ids': 4,
   'wf_centres': [3.790737, -0.209912, 3.776438],
   'wf_spreads': 1.34235371},
  {'wf_ids': 5,
   'wf_centres': [3.612413, 0.523184, 4.229709],
   'wf_spreads': 1.34157469},
  {'wf_ids': 6,
   'wf_centres': [3.700439, -0.109756, 4.641866],
   'wf_spreads': 1.30361953},
  {'wf_ids': 7,
   'wf_centres': [2.786112, 2.305423, 1.595865],
   'wf_spreads': 1.31444078},
  {'wf_ids': 8,
   'wf_centres': [3.40987

In [8]:
# Here are some important numbers for the calculation.
result = {
    # This is the number of Wannier functions
    "number_wfs": out_dict['number_wfs'],
    # These are the spreads of this calculations
    # Omega_I, Omega_D, and Omega_OD are three components of the total Omega
    # It would be great if we can show these numbers on materialscloud,
    # as they are important for checking the quality of Wannier functions.
    "Omega_I": out_dict['Omega_I'],
    "Omega_D": out_dict['Omega_D'],
    "Omega_OD": out_dict['Omega_OD'],
    "Omega": out_dict['Omega_I'] + out_dict['Omega_D'] + out_dict['Omega_OD']
}
print(result)

{'number_wfs': 14, 'Omega_I': 16.310849001, 'Omega_D': 0.026051159, 'Omega_OD': 2.331212402, 'Omega': 18.668112562}


In [9]:
# Each Wannier function has a center (vector of three floats),
# and a spread (a float), which means the spatial extent of the Wannier function.
# Ideally, these can be shown as a table on materialscloud.
data = []
for wf in out_dict["wannier_functions_output"]:
    data.append({
        "index": wf["wf_ids"],
        "center": wf["wf_centres"],
        "spread": wf["wf_spreads"],
    })
print(tabulate(data, headers="keys"))

  index  center                              spread
-------  --------------------------------  --------
      1  [1.38448, 4.069295, 1.773057]      1.32289
      2  [-0.919722, 2.009378, 4.143696]    1.32296
      3  [4.414502, 0.245519, 4.320885]     1.31446
      4  [3.790737, -0.209912, 3.776438]    1.34235
      5  [3.612413, 0.523184, 4.229709]     1.34157
      6  [3.700439, -0.109756, 4.641866]    1.30362
      7  [2.786112, 2.305423, 1.595865]     1.31444
      8  [3.409876, 1.849973, 2.140296]     1.34235
      9  [3.588185, 2.583099, 1.687087]     1.34154
     10  [3.5002, 1.950199, 1.274867]       1.30357
     11  [0.378268, 1.038758, -0.042469]    1.345
     12  [0.040758, 1.622904, 0.453888]     1.36445
     13  [-0.378084, 1.037984, 0.042725]    1.34463
     14  [-0.040943, 1.622626, -0.454146]   1.36428


In [10]:
# It would be great if we can show a comparison of band structures between DFT
# and Wannier90 on materialscloud, as in `bandsdiff_InN_5532417_13171598.pdf`.
# The BandsData for DFT and Wannier90 can be accessed in the previous
# `band_pw` and `band_w90`, respectively.

In [11]:
# The real-space shape of each Wannier function is stored in a cube
# file (e.g. aiida_00001.cube) in the wan_tb_cube.7z file.
# If possible, it would be great if we can render these 3D volume data as
# isosurfaces on materialscloud.
# But we can postpone this to the future if this is difficult.
# For now, we can just add a button to download the wan_tb_cube.7z file.